In [1]:
import torch 
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
path = 'dataset'
dataset = pq.ParquetDataset(path)

In [4]:
dataset.schema

X_jet: list<item: list<item: list<item: double>>>
  child 0, item: list<item: list<item: double>>
      child 0, item: list<item: double>
          child 0, item: double
m: double
iphi: double
pt: double
ieta: double

In [5]:
dataset.filesystem,dataset.files

(<pyarrow._fs.LocalFileSystem at 0x26612b81630>,
 ['dataset/top_gun_opendata_0.parquet',
  'dataset/top_gun_opendata_1.parquet',
  'dataset/top_gun_opendata_3.parquet',
  'dataset/top_gun_opendata_4.parquet',
  'dataset/top_gun_opendata_5.parquet',
  'dataset/top_gun_opendata_6.parquet'])

In [6]:
dataset = pq.ParquetDataset(path)

for file_path in dataset.files:
    f = pq.ParquetFile(file_path)
    print(f"File: {file_path}")
    print(f.schema_arrow)
    print()


File: dataset/top_gun_opendata_0.parquet
X_jet: list<item: list<item: list<item: double>>>
  child 0, item: list<item: list<item: double>>
      child 0, item: list<item: double>
          child 0, item: double
m: double
iphi: double
pt: double
ieta: double

File: dataset/top_gun_opendata_1.parquet
X_jet: list<item: list<item: list<item: double>>>
  child 0, item: list<item: list<item: double>>
      child 0, item: list<item: double>
          child 0, item: double
m: double
iphi: double
pt: double
ieta: double

File: dataset/top_gun_opendata_3.parquet
X_jet: list<item: list<item: list<item: double>>>
  child 0, item: list<item: list<item: double>>
      child 0, item: list<item: double>
          child 0, item: double
m: double
iphi: double
pt: double
ieta: double

File: dataset/top_gun_opendata_4.parquet
X_jet: list<item: list<item: list<item: double>>>
  child 0, item: list<item: list<item: double>>
      child 0, item: list<item: double>
          child 0, item: double
m: double
ip

In [7]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import pyarrow.parquet as pq
from torch.utils.data import Dataset, DataLoader, random_split

class ParquetFileDataset(Dataset):
    def __init__(self, parquet_file, x_col='X_jet', y_col='m', max_jets=4, jet_h=125, jet_w=125):
        self.parquet_file = parquet_file
        self.x_col = x_col
        self.y_col = y_col
        self.max_jets = max_jets
        self.jet_h = jet_h
        self.jet_w = jet_w
        
        # Load ONLY this file into RAM
        table = pq.read_table(self.parquet_file, columns=[self.x_col, self.y_col])
        self.df = table.to_pandas()

        x_list = []
        for x_val in self.df[self.x_col]:
            x = np.asarray(x_val, dtype=np.float32)
            if x.shape[0] > self.max_jets:
                x = x[:self.max_jets]
            elif x.shape[0] < self.max_jets:
                pad = np.zeros((self.max_jets - x.shape[0], self.jet_h, self.jet_w), dtype=np.float32)
                x = np.concatenate([x, pad], axis=0)
            x_list.append(x)

        self.X = torch.from_numpy(np.stack(x_list, axis=0)).contiguous()
        self.y = torch.from_numpy(self.df[self.y_col].to_numpy(dtype=np.float32)).contiguous()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def get_parquet_files(folder_path, max_files=5):
    files = sorted(glob.glob(os.path.join(folder_path, '*.parquet')))
    if not files:
        raise FileNotFoundError(f'No parquet files found at: {folder_path}')
    return files[:max_files]

def create_loaders_for_file(file_path, batch_size=16, val_split=0.2, seed=42, num_workers=0):
    """Creates dataset and loaders for a SINGLE file."""
    dataset = ParquetFileDataset(file_path)
    n_total = len(dataset)
    n_train = int((1 - val_split) * n_total)
    n_val = n_total - n_train
    if n_train == 0 and n_total > 0:
        n_train = 1
        n_val = n_total - 1

    train_dataset, val_dataset = random_split(
        dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(seed)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )
    
    return {
        'file_path': file_path,
        'train_loader': train_loader,
        'val_loader': val_loader,
        'train_size': len(train_dataset),
        'val_size': len(val_dataset),
    }

## Why this `__getitem__` is faster

- The old version did `read_row_group(...)` for almost every sample, which caused heavy repeated disk reads.
- The new version maps each global index to a `(file, row_group, row)` tuple using cumulative real row counts.
- It loads an entire row group once, converts it once, and caches it in an LRU cache.
- Repeated accesses to nearby samples reuse in-memory tensors instead of hitting disk again.
- This reduces both disk I/O and Python/PyArrow conversion overhead, which is why latency drops significantly.

In [8]:
import os
import time
import glob
import numpy as np
import pyarrow.parquet as pq

candidate_dirs = []

if "path" in globals() and isinstance(path, str) and len(path) > 0:
    if path.endswith(".parquet"):
        candidate_dirs.append(os.path.dirname(path))
    else:
        candidate_dirs.append(path)

for parquet_file in glob.glob("**/*.parquet", recursive=True):
    candidate_dirs.append(os.path.dirname(parquet_file))

# Keep order but remove duplicates
candidate_dirs = list(dict.fromkeys(candidate_dirs))

def folder_has_rows(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    if not files:
        return False
    total = 0
    for fp in files:
        pf = pq.ParquetFile(fp)
        md = pf.metadata
        for rg_idx in range(md.num_row_groups):
            total += md.row_group(rg_idx).num_rows
            if total > 0:
                return True
    return False

dataset_path = None
for folder in candidate_dirs:
    if folder_has_rows(folder):
        dataset_path = folder
        break

if dataset_path is None:
    raise FileNotFoundError("No parquet folder with rows found. Set `path` to your parquet directory.")

jet_dataset = JetDataset(dataset_path, max_cached_row_groups=16)

# Warm-up: first access may include cold disk read + decode
_ = jet_dataset[0]

indices = np.random.randint(0, len(jet_dataset), size=200)
start = time.time()
for i in indices:
    _ = jet_dataset[int(i)]
elapsed = time.time() - start

print(f"Dataset path: {dataset_path}")
print(f"Dataset size: {len(jet_dataset)}")
print(f"Avg __getitem__ over 200 random samples: {elapsed/len(indices):.6f} s")
print(f"Total for 200 samples: {elapsed:.3f} s")

NameError: name 'JetDataset' is not defined

In [ ]:
os.cpu_count()

16

In [ ]:
import os
import torch

# Speed-focused settings
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

dataset_path = path

# Windows multiprocessing safe-guard
if os.name == 'nt':
    num_workers = 0
else:
    num_workers = 0 # 0 avoids multiprocessing overhead for purely in-memory data

batch_size = 128
max_files_count = 5

# 1. Just identify files first (no heavy loading yet)
all_files = get_parquet_files(dataset_path, max_files=max_files_count)
print(f"Found {len(all_files)} files to train on (sequentially).")

# Just peek at the first file to confirm it works (optional)
print(f"Peeking at first file metadata: {os.path.basename(all_files[0])}")
peek_loader = create_loaders_for_file(all_files[0], batch_size, num_workers=num_workers)
print(f"  train: {peek_loader['train_size']}, val: {peek_loader['val_size']}")
# Let it go out of scope immediately to free memory
del peek_loader

Using 8 dataloader workers. Batch size: 128
Train samples: 721172, Val samples: 180294


In [ ]:
#just checking if hte dataset is working as expected 
jet_dataset = JetDataset(path)
x, y = jet_dataset[0]
print(x.shape, x.dtype, y.shape, y.dtype)

torch.Size([4, 125, 125]) torch.float32 torch.Size([]) torch.float32


In [ ]:
! nvidia-smi

Mon Mar  2 17:24:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.74                 Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   66C    P0             20W /   95W |    3454MiB /   6141MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import numpy as np
import itertools
from models import convnet
import gc # Import garbage collector to force cleanup

architectures = {
    "54_layers": [4, 6, 12, 5],
    "98_layers": [8, 12, 21, 8]
}

learning_rates = [1e-3, 5e-4]
weight_decays = [1e-4, 0.0]

class MassRegressionNet(nn.Module):
    def __init__(self, layers_config):
        super().__init__()
        self.backbone = convnet(layers_config, inplanes=4)
        
        dummy_input = torch.zeros(1, 4, 125, 125)
        with torch.no_grad():
            out = self.backbone(dummy_input)
            flattened_size = out.flatten(1).shape[1]
            
        self.regressor = nn.Sequential(
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return self.regressor(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Number of GPUs available: {torch.cuda.device_count()}")

criterion = nn.MSELoss()

train_transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5)
])

epochs = 1
perform_training = True

# Dictionary to store the loss history within memory
training_history = {}

try:
    all_files = get_parquet_files(path, max_files=max_files_count)
except NameError:
    all_files = [] # just in case path isn't defined

if len(all_files) == 0:
    print("No files found!")

for arch_name, config in architectures.items():
    for lr, wd in itertools.product(learning_rates, weight_decays):
        print(f"Initializing/Training Architecture: {arch_name} | LR: {lr} | WD: {wd}")
        
        model = MassRegressionNet(config)
        
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs for training!")
            model = nn.DataParallel(model)
            
        model = model.to(device)
        print('model parameters: ',np.sum([p.numel() for p in model.parameters()]))
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        
        train_losses = []
        val_losses = []

        if perform_training:
            for epoch in range(epochs):
                model.train()
                total_train_loss = 0.0
                total_train_samples = 0

                # Sequentially load one file, train, then release memory
                for i, file_path in enumerate(all_files, start=1):
                    # Create loader for this specific file only
                    # print(f"  [RAM] Loading file {i}/{len(all_files)}: {os.path.basename(file_path)}")
                    file_data = create_loaders_for_file(file_path, batch_size=batch_size, num_workers=num_workers)
                    
                    train_loader = file_data['train_loader']
                    val_loader = file_data['val_loader']

                    # Train on this file's data
                    for images, targets in train_loader:
                        images = train_transforms(images).to(device, non_blocking=True)
                        targets = targets.to(device, non_blocking=True)
                        
                        optimizer.zero_grad()
                        outputs = model(images)
                        loss = criterion(outputs.squeeze(), targets)
                        loss.backward()
                        optimizer.step()
                        
                        total_train_loss += loss.item() * images.size(0)
                        total_train_samples += images.size(0)
                    
                    # Release memory immediately after file is processed
                    del train_loader
                    del val_loader
                    del file_data
                    gc.collect() # Force garbage collection
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                train_loss = total_train_loss / total_train_samples if total_train_samples > 0 else 0
                
                # Validation phase (optional: can also run purely on last file or a specific val set. 
                # Here we re-load files but strictly for validation to get true epoch val loss? 
                # Or simpler: accumulate val loss during the training pass above?
                # But standard practice is separate pass. To save time, let's just use the LAST file for validation check
                # or re-load a small subset. For correctness, let's re-run validation on ALL files efficiently.
                
                model.eval()
                total_val_loss = 0.0
                total_val_samples = 0
                
                with torch.no_grad():
                    for file_path in all_files:
                         # Load just for validation pass
                        file_data = create_loaders_for_file(file_path, batch_size=batch_size, num_workers=num_workers)
                        val_loader = file_data['val_loader']
                        
                        for val_images, val_targets in val_loader:
                            val_images = val_images.to(device)
                            val_targets = val_targets.to(device)
                            
                            val_outputs = model(val_images)
                            v_loss = criterion(val_outputs.squeeze(), val_targets)
                            total_val_loss += v_loss.item() * val_images.size(0)
                            total_val_samples += val_images.size(0)
                        
                        del val_loader
                        del file_data
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                test_loss = total_val_loss / total_val_samples if total_val_samples > 0 else 0
                
                train_losses.append(train_loss)
                val_losses.append(test_loss)
                
                print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {test_loss:.4f}")
                
        history_key = f"model_{arch_name}_lr_{lr}_wd_{wd}"
        training_history[history_key] = {
            'train_losses': train_losses,
            'val_losses': val_losses
        }
                
        model_filename = f"convnet_{arch_name}_lr_{lr}_wd_{wd}.pth"
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), model_filename)
        else:
            torch.save(model.state_dict(), model_filename)
            
        print(f"Model Saved: {model_filename}\n{'-'*50}\n")

Using device: cuda
Number of GPUs available: 1
Initializing/Training Architecture: 54_layers | LR: 0.001 | WD: 0.0001
model parameters:  30344281
batchno0 : running_loss is 13919130.0
batchno10 : running_loss is 80591270.875
batchno20 : running_loss is 99824644.125
batchno30 : running_loss is 117117556.5
batchno40 : running_loss is 133648959.625
batchno50 : running_loss is 158257760.25
batchno60 : running_loss is 174049493.75
batchno70 : running_loss is 190542750.25
batchno80 : running_loss is 205673131.625
batchno90 : running_loss is 221301118.75
batchno100 : running_loss is 236031878.875
batchno110 : running_loss is 253127253.125


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001676B22D9A0>>
Traceback (most recent call last):
  File "c:\Users\visha\anaconda3\envs\learning_pytorch\Lib\site-packages\ipykernel\ipkernel.py", line 790, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\visha\anaconda3\envs\learning_pytorch\Lib\threading.py", line 1535, in enumerate
    def enumerate():
    
KeyboardInterrupt: 
